# 04 — Drift Analysis

Computes the **temporal diagnostic profile** for each window pair (A, B) across all
supported model types, then renders a combined dashboard. Reflects the methodology
described in the thesis (no drift ratio; per-window baselines reported separately
for A and B; global attribution drift restricted to SHAP).

**Model types:** `MODEL_TYPES = ['xgboost', 'logreg', 'mlp_plr']`
Artifacts are read from `data/models/{model_type}/`, `data/shap/{model_type}/`, and
`data/lime/{model_type}/`.

**Input:** `data/processed/`, `data/windows/`, `data/models/{model_type}/`, `data/shap/{model_type}/`, `data/lime/{model_type}/`
**Output per model type:** `data/results/{model_type}/drift_metrics.csv`
**Combined outputs:** `data/results/combined/drift_metrics_combined.csv`, `data/results/combined/drift_dashboard_*.png`

---

| Metric | Symbol | XGBoost (SHAP) | XGBoost / MLP-PLR (LIME) | LR (coef ref) |
|---|---|---|---|---|
| Covariate drift | Δ_X | ✓ | ✓ | ✓ |
| Target drift | Δ_Y | ✓ | ✓ | ✓ |
| Performance change | Δ_perf | ✓ | ✓ | ✓ |
| Local dynamic drift (cosine) | loc_cos | ✓ (SHAP) | ✓ (LIME) | — |
| Local dynamic drift (RBO) | loc_rbo | ✓ (SHAP) | ✓ (LIME) | — |
| Within-window baseline (cosine) — A | base_cos_A | ✓ | ✓ | ✓ |
| Within-window baseline (cosine) — B | base_cos_B | ✓ | ✓ | ✓ |
| Within-window baseline (RBO) — A | base_rbo_A | ✓ | ✓ | ✓ |
| Within-window baseline (RBO) — B | base_rbo_B | ✓ | ✓ | ✓ |
| Global attribution drift | global_rbo | ✓ (SHAP only) | — | — |
| Explainer stochasticity | `explainer_stoch_max_abs` / `explainer_stoch_median_abs`  | ✓ | ✓ | N/A |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

> **Setup note:** this notebook requires the `rbo` package for rank-biased overlap metrics (§3.5, §3.8).
> If not already installed, run: `pip install rbo`

In [ ]:
%pip install rbo --no-deps

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pathlib import Path
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import rbo

WORKSPACE  = Path('/content/drive/MyDrive/HomeCredit_workspace')
PROC_DIR   = WORKSPACE / 'data' / 'processed'
WIN_DIR    = WORKSPACE / 'data' / 'windows'

# ── Per-model-type base directories ───────────────────────────────────
MODEL_DIR_BASE   = WORKSPACE / 'data' / 'models'
SHAP_DIR_BASE    = WORKSPACE / 'data' / 'shap'
LIME_DIR_BASE    = WORKSPACE / 'data' / 'lime'
RESULTS_DIR_BASE = WORKSPACE / 'data' / 'results'

# Aliases used by downstream cells
SHAP_DIR    = SHAP_DIR_BASE / 'xgboost'
RESULTS_DIR = RESULTS_DIR_BASE

COMBINED_DIR = RESULTS_DIR_BASE / 'combined'
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

# ── What to compute ────────────────────────────────────────────────────
MODEL_TYPES = ['xgboost', 'logreg', 'mlp_plr']
EXPLAINERS  = ['shap', 'lime']   # per-row attribution sources to iterate over
                                 # LR emits a 'coef' row handled outside this list

RBO_P = 0.9

print('Imports OK')
print(f'MODEL_TYPES: {MODEL_TYPES}')
print(f'EXPLAINERS : {EXPLAINERS}')

In [ ]:
X    = pd.read_parquet(PROC_DIR / 'X.parquet').values.astype(np.float32)
Y    = np.load(PROC_DIR / 'Y.npy').astype(np.int8)

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names_json = json.load(f)
feature_names     = feature_names_json['all']
num_feature_names = feature_names_json['num']

# Column indices of numeric / binary features in X (column order matches feature_names)
num_col_idx = [feature_names.index(fn) for fn in num_feature_names]
bin_col_idx = [i for i in range(len(feature_names)) if i not in set(num_col_idx)]

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)

R     = config['parameters']['R']
pairs = config['pairs']
n_features = len(feature_names)

assert X.shape[0] == Y.shape[0], "X and Y have inconsistent numbers of rows."
assert X.shape[1] == n_features, "X column count does not match feature_names['all']."
assert len(set(feature_names)) == len(feature_names), "feature_names contains duplicates."
assert len(set(num_col_idx) & set(bin_col_idx)) == 0, "Numeric and binary indices overlap."
assert len(num_col_idx) + len(bin_col_idx) == n_features, (
    "Numeric/binary feature partition does not cover all features."
)
assert len(pairs) > 0, "No rolling-window pairs found in window_config.json."

print(f'X: {X.shape}, features: {n_features}, R={R}, pairs: {len(pairs)}')
print(f'  Numeric: {len(num_col_idx)}, Binary: {len(bin_col_idx)}')

## Distance and RBO helper functions

In [ ]:
def cosine_distance(u: np.ndarray, v: np.ndarray) -> float:
    """d_cos(u,v) = 1 - cosine_similarity(u,v).
    Both-zero → 0.0; one-zero one-nonzero → np.nan (excluded by nanmedian aggregation).
    """
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u < 1e-12 and norm_v < 1e-12:
        return 0.0
    if norm_u < 1e-12 or norm_v < 1e-12:
        # One attribution vector is exactly zero; the other is not.
        # Not a meaningful distance — marked nan and dropped during aggregation.
        return np.nan
    return float(1.0 - np.dot(u, v) / (norm_u * norm_v))


def rbo_score_local(l1, l2, p=0.9):
    """Simple RBO implementation as fallback."""
    if not l1 or not l2: return 0.0
    n = min(len(l1), len(l2))
    s1, s2 = set(), set()
    agreement = 0.0
    res = 0.0
    for i in range(n):
        s1.add(l1[i])
        s2.add(l2[i])
        agreement = len(s1.intersection(s2)) / (i + 1)
        res += agreement * (p ** i)
    return res * (1 - p)


def rbo_distance(u: np.ndarray, v: np.ndarray, p: float = RBO_P) -> float:
    """
    d_rbo(u, v) = 1 - RBO(rank(u), rank(v)).
    Features are ranked by decreasing absolute attribution.
    Both-zero → 0.0; one-zero one-nonzero → np.nan.
    """
    u_zero = np.all(np.abs(u) < 1e-12)
    v_zero = np.all(np.abs(v) < 1e-12)
    if u_zero and v_zero:
        return 0.0
    if u_zero or v_zero:
        return np.nan
    rank_u = list(np.argsort(-np.abs(u)))
    rank_v = list(np.argsort(-np.abs(v)))
    score = rbo.RankingSimilarity(rank_u, rank_v).rbo(p=p)
    return float(1.0 - score)


def instance_dynamic_drift(
    phi_A: np.ndarray,  # (R, p) — R attribution vectors from window A
    phi_B: np.ndarray,  # (R, p) — R attribution vectors from window B
    dist_fn,
) -> float:
    """
    δ_dyn(x; d) = mean over all R×R cross-window pairs of d(φ_A^{(r)}, φ_B^{(s)}).
    """
    dists = []
    for r in range(R):
        for s in range(R):
            dists.append(dist_fn(phi_A[r], phi_B[s]))
    return float(np.nanmean(dists))


def instance_baseline_instability(
    phi: np.ndarray,  # (R, p) — R attribution vectors from same window
    dist_fn,
) -> float:
    """
    δ^{base}(x; d) = median over all r < r' pairs of d(φ^{(r)}, φ^{(r')}).
    """
    dists = []
    for r, r2 in combinations(range(R), 2):
        dists.append(dist_fn(phi[r], phi[r2]))
    return float(np.nanmedian(dists)) if dists else 0.0


def rbo_global_drift(
    phi_bar_A: np.ndarray,  # (|F|, p)
    phi_bar_B: np.ndarray,  # (|F|, p)
    p: float = RBO_P,
) -> float:
    """
    Δ_E^{glob}: 1 − RBO(rank(g_A), rank(g_B)) where g(j) = mean_x |φ̄_j(x)|.
    """
    g_A = np.abs(phi_bar_A).mean(axis=0)
    g_B = np.abs(phi_bar_B).mean(axis=0)
    rank_A = list(np.argsort(-g_A))
    rank_B = list(np.argsort(-g_B))
    score = rbo.RankingSimilarity(rank_A, rank_B).rbo(p=p)
    return float(1.0 - score)


print('Distance functions defined.')

## Main drift computation loop

For each window pair we compute all metrics and append to a results list.

### Logistic Regression — reference treatment

LR is treated as a transparent reference, *not* as a post-hoc explanation drift
stream. Two concrete artefacts are used:

- **Replica coefficient tensors** (`coef_A.npy`, `coef_B.npy`, shape `(R, p)`)
  are used only for within-window coefficient instability under cosine and RBO
  distance. This estimates the variability induced by the retraining procedure
  for the linear model and plays the same role as `base_*_A` / `base_*_B`
  for SHAP and LIME.
- **Full-window coefficient vectors** (`coef_A_full.npy`, `coef_B_full.npy`,
  shape `(p,)`) are obtained from a single deterministic LR fit on the full
  training window, with no bootstrap and no replica averaging. These are
  visualised side by side as a transparent A-vs-B reference (see the
  full-window coefficient comparison cell below).

| Attribution metric              | LR analogue                                                   | Notes                                       |
|---------------------------------|---------------------------------------------------------------|---------------------------------------------|
| Covariate drift Δ_X             | ✓ identical                                                   | data-level, model-agnostic                  |
| Target drift Δ_Y                | ✓ identical                                                   | data-level, model-agnostic                  |
| Performance change Δ_perf       | ✓ identical                                                   | same PR-AUC formula                         |
| `loc_cos` / `loc_rbo`           | — *(not computed)*                                            | replica-coefficient cross-window drift conflates retraining noise with the per-replica scaler basis; not reported |
| `base_cos_A/B`, `base_rbo_A/B`  | median pairwise distance over C(R,2) replica coefficient pairs per window | within-window coefficient instability       |
| Global attribution drift        | — *(not computed)*                                            | LR coefficients are global by construction; covered by the full-window comparison plot |
| `explainer_stoch_max_abs` / `explainer_stoch_median_abs`              | NaN                                                           | LR coefficient solvers are deterministic    |

In [ ]:
all_results = {}


def _load_attributions(explainer_name, pair_dir, attr_dir, pred_data):
    """Return (attr_A, attr_B, flagged_subset_idx_within_flagged) or None.

    For SHAP:
    - XGBoost usually runs on the full flagged set, so no subset file exists.
    - MLP-PLR may run on a saved SHAP subsample. If
      mlp_shap_subsample_idx.npy exists, it is used as the subset index.
      Otherwise, SHAP is treated as full-flagged for backwards compatibility.

    For LIME:
    - attr files are (R, |F_lime|, p);
    - lime_flagged_idx.npy stores positions within the full flagged set.
    """
    if explainer_name == 'shap':
        a_path = attr_dir / 'shap_A.npy'
        b_path = attr_dir / 'shap_B.npy'
        subset_path = attr_dir / 'mlp_shap_subsample_idx.npy'

        if not (a_path.exists() and b_path.exists()):
            return None

        a = np.load(a_path)
        b = np.load(b_path)

        if subset_path.exists():
            subset = np.load(subset_path).astype(np.int64)
        else:
            subset = np.arange(a.shape[1], dtype=np.int64)

        return a, b, subset

    elif explainer_name == 'lime':
        a_path      = attr_dir / 'lime_A.npy'
        b_path      = attr_dir / 'lime_B.npy'
        subset_path = attr_dir / 'lime_flagged_idx.npy'

        if not (a_path.exists() and b_path.exists() and subset_path.exists()):
            return None

        a = np.load(a_path)
        b = np.load(b_path)
        subset = np.load(subset_path).astype(np.int64)

        return a, b, subset

    raise ValueError(f'Unknown explainer: {explainer_name}')


def _validate_attribution_shapes(attr_A, attr_B, subset, flagged_local_idx,
                                 mt, explainer_name, pid):
    """Fail early if attribution arrays are stale or incompatible."""
    assert attr_A.ndim == 3 and attr_B.ndim == 3, (
        f'{mt}/{explainer_name}/pair {pid:02d}: attribution arrays must be 3-D.'
    )

    assert attr_A.shape == attr_B.shape, (
        f'{mt}/{explainer_name}/pair {pid:02d}: attr_A and attr_B shape mismatch: '
        f'{attr_A.shape} vs {attr_B.shape}'
    )

    assert attr_A.shape[0] == R, (
        f'{mt}/{explainer_name}/pair {pid:02d}: expected R={R}, '
        f'got attribution R={attr_A.shape[0]}'
    )

    assert attr_A.shape[2] == n_features, (
        f'{mt}/{explainer_name}/pair {pid:02d}: expected {n_features} features, '
        f'got {attr_A.shape[2]}'
    )

    assert len(subset) == attr_A.shape[1], (
        f'{mt}/{explainer_name}/pair {pid:02d}: subset length {len(subset)} '
        f'does not match attribution instances {attr_A.shape[1]}'
    )

    assert np.all((subset >= 0) & (subset < len(flagged_local_idx))), (
        f'{mt}/{explainer_name}/pair {pid:02d}: subset indices are outside flagged_idx.'
    )


def _validate_predictions(pred_data, mt, pid, idx_eval):
    """Check that predictions.npz contains the keys and shapes expected by notebook 04."""
    required_keys = [
        'preds_A', 'preds_B',
        'p_hat_A', 'p_hat_B',
        'flagged_idx',
        'Y_eval',
        'pr_auc_A', 'pr_auc_B',
        'per_replica_pr_auc_A', 'per_replica_pr_auc_B',
    ]

    missing = [k for k in required_keys if k not in pred_data.files]
    assert not missing, (
        f'{mt}/pair {pid:02d}: predictions.npz missing keys: {missing}. '
        'Delete stale predictions and rerun the training notebook.'
    )

    flagged_idx = pred_data['flagged_idx']
    assert pred_data['preds_A'].shape == pred_data['preds_B'].shape, (
        f'{mt}/pair {pid:02d}: preds_A and preds_B shape mismatch.'
    )

    assert pred_data['preds_A'].shape[0] == R, (
        f'{mt}/pair {pid:02d}: expected R={R}, got {pred_data["preds_A"].shape[0]}.'
    )

    assert pred_data['preds_A'].shape[1] == len(idx_eval), (
        f'{mt}/pair {pid:02d}: prediction length does not match idx_eval.'
    )

    assert np.all((flagged_idx >= 0) & (flagged_idx < len(idx_eval))), (
        f'{mt}/pair {pid:02d}: flagged_idx contains positions outside idx_eval.'
    )


def _attribution_drift_metrics(attr_A: np.ndarray, attr_B: np.ndarray) -> dict:
    """Compute local drift, separate A/B baselines, and global drift.

    Returns keys: loc_cos, loc_rbo, base_cos_A, base_cos_B, base_rbo_A,
    base_rbo_B, global_rbo. The caller sets global_rbo to NaN for LIME.
    """
    n_sub = attr_A.shape[1]
    out   = {}

    # Local dynamic cross-window drift
    dyn_cos_per_instance = []
    dyn_rbo_per_instance = []
    for i in range(n_sub):
        phi_A_i = attr_A[:, i, :]
        phi_B_i = attr_B[:, i, :]
        dyn_cos_per_instance.append(instance_dynamic_drift(phi_A_i, phi_B_i, cosine_distance))
        dyn_rbo_per_instance.append(instance_dynamic_drift(phi_A_i, phi_B_i, rbo_distance))
    out['loc_cos'] = float(np.nanmedian(dyn_cos_per_instance))
    out['loc_rbo'] = float(np.nanmedian(dyn_rbo_per_instance))

    # Within-window baseline instability — reported separately for A and B
    base_cos_A_per, base_cos_B_per = [], []
    base_rbo_A_per, base_rbo_B_per = [], []
    for i in range(n_sub):
        phi_A_i = attr_A[:, i, :]
        phi_B_i = attr_B[:, i, :]
        base_cos_A_per.append(instance_baseline_instability(phi_A_i, cosine_distance))
        base_cos_B_per.append(instance_baseline_instability(phi_B_i, cosine_distance))
        base_rbo_A_per.append(instance_baseline_instability(phi_A_i, rbo_distance))
        base_rbo_B_per.append(instance_baseline_instability(phi_B_i, rbo_distance))
    out['base_cos_A'] = float(np.nanmedian(base_cos_A_per))
    out['base_cos_B'] = float(np.nanmedian(base_cos_B_per))
    out['base_rbo_A'] = float(np.nanmedian(base_rbo_A_per))
    out['base_rbo_B'] = float(np.nanmedian(base_rbo_B_per))

    # Global attribution drift (SHAP only — caller NaNs this for LIME)
    phi_bar_A = attr_A.mean(axis=0)
    phi_bar_B = attr_B.mean(axis=0)
    out['global_rbo'] = rbo_global_drift(phi_bar_A, phi_bar_B)

    return out


def _make_base_row(mt, pid, p, pred_data, idx_A, idx_B, flagged_local_idx):
    """Per-pair metrics that are explainer-independent (data-drift / target-drift / perf)."""
    row = {
        'model_type':    mt,
        'explainer':     None,   # filled in later
        'pair_id':       pid,
        'step_label_A':  p['step_label_A'],
        'step_label_B':  p['step_label_B'],
        'n_train_A':     p['n_train_A'],
        'n_train_B':     p['n_train_B'],
        'n_eval':        p['n_eval'],
        'n_flagged':     len(flagged_local_idx),
        'pr_auc_A':      float(pred_data['pr_auc_A']),
        'pr_auc_B':      float(pred_data['pr_auc_B']),
    }

    # Performance change: PR-AUC_B − PR-AUC_A (positive = newer model better)
    row['delta_perf'] = float(pred_data['pr_auc_B']) - float(pred_data['pr_auc_A'])

    # Target drift (binary): absolute difference in positive-class rate
    row['delta_Y'] = abs(float(Y[idx_A].mean()) - float(Y[idx_B].mean()))

    # Covariate drift: feature-wise W1 in the reference frame, averaged
    pair_dir = MODEL_DIR_BASE / mt / f'pair_{pid:02d}'
    reference_scaler_path = pair_dir / 'reference_scaler.joblib'
    if reference_scaler_path.exists():
        reference_scaler = joblib.load(reference_scaler_path)
        X_A_num_sc = reference_scaler.transform(X[idx_A][:, num_col_idx])
        X_B_num_sc = reference_scaler.transform(X[idx_B][:, num_col_idx])
        w1_per_feat = np.zeros(n_features, dtype=np.float64)
        for k, j in enumerate(num_col_idx):
            w1_per_feat[j] = wasserstein_distance(X_A_num_sc[:, k], X_B_num_sc[:, k])
        for j in bin_col_idx:
            w1_per_feat[j] = wasserstein_distance(X[idx_A][:, j], X[idx_B][:, j])
        row['delta_X'] = float(w1_per_feat.mean())
    else:
        row['delta_X'] = np.nan

    return row


def _fmt(v, w=6, p=4):
    """Per-pair log: '-' for NaN/None, fixed-width float otherwise."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return f'{"-":>{w}s}'
    return f'{v:{w}.{p}f}'


def _fmt_sci(v, w=8, p=2):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return f'{"-":>{w}s}'
    return f'{v:{w}.{p}e}'


# ═════════════════════════════════════════════════════════════════════════════
# Main loop
# ═════════════════════════════════════════════════════════════════════════════

for mt in MODEL_TYPES:
    print(f'\n{"="*60}')
    print(f'  Model type: {mt}')
    print(f'{"="*60}')

    mt_model_dir = MODEL_DIR_BASE / mt
    results = []

    for p in pairs:
        pid      = p['pair_id']
        pair_dir = mt_model_dir / f'pair_{pid:02d}'

        print(f'\n── [{mt}] Pair {pid:02d}: A_end={p["step_label_A"]}  B_end={p["step_label_B"]} ──')

        pred_data_path = pair_dir / 'predictions.npz'
        if not pred_data_path.exists():
            print(f'  WARNING: predictions.npz not found for pair {pid:02d}. Skipping.')
            continue

        pred_data = np.load(pred_data_path)
        idx_A    = np.array(p['idx_A'],    dtype=np.int64)
        idx_B    = np.array(p['idx_B'],    dtype=np.int64)
        idx_eval = np.array(p['idx_eval'], dtype=np.int64)
        _validate_predictions(pred_data, mt, pid, idx_eval)

        flagged_local_idx = pred_data['flagged_idx']
        n_flagged = len(flagged_local_idx)
        print(f'  Flagged: {n_flagged:,}')

        base_row = _make_base_row(mt, pid, p, pred_data, idx_A, idx_B, flagged_local_idx)
        print(f'  Perf: Δ={base_row["delta_perf"]:+.4f}   Target drift: {base_row["delta_Y"]:.4f}   Cov drift: {base_row["delta_X"]:.4f}')

        # ─────────────────────────────────────────────────────────────
        # LR reference path — within-window coefficient instability only.
        # These distances are computed on the saved replica coefficient vectors.
        # Because each replica includes its own fitted scaler, this should be read as
        # LR pipeline-level coefficient instability rather than pure coefficient
        # variation in one fixed standardized basis.
        # ─────────────────────────────────────────────────────────────
        if mt == 'logreg':
            coef_A_path = pair_dir / 'coef_A.npy'
            coef_B_path = pair_dir / 'coef_B.npy'
            if coef_A_path.exists() and coef_B_path.exists():
                coef_A = np.load(coef_A_path)
                coef_B = np.load(coef_B_path)

                row = dict(base_row)
                row['explainer']                   = 'coef'
                row['n_subset']                    = R   # one coef vector per replica
                row['shap_stochasticity']          = np.nan
                row['explainer_stoch_max_abs']     = np.nan
                row['explainer_stoch_median_abs']  = np.nan
                row['explainer_is_deterministic']  = True

                # Cross-window drift columns are intentionally NaN for LR.
                row['loc_cos']    = np.nan
                row['loc_rbo']    = np.nan
                row['global_rbo'] = np.nan

                # Within-window coefficient instability — separate for A and B.
                row['base_cos_A'] = float(np.nanmedian([cosine_distance(coef_A[r], coef_A[r2])
                                                        for r, r2 in combinations(range(R), 2)]))
                row['base_cos_B'] = float(np.nanmedian([cosine_distance(coef_B[r], coef_B[r2])
                                                        for r, r2 in combinations(range(R), 2)]))
                row['base_rbo_A'] = float(np.nanmedian([rbo_distance(coef_A[r], coef_A[r2])
                                                        for r, r2 in combinations(range(R), 2)]))
                row['base_rbo_B'] = float(np.nanmedian([rbo_distance(coef_B[r], coef_B[r2])
                                                        for r, r2 in combinations(range(R), 2)]))

                print(f'  [coef]  cos: loc={_fmt(np.nan)}  base_A={_fmt(row["base_cos_A"])}  base_B={_fmt(row["base_cos_B"])}')
                print(f'          rbo: loc={_fmt(np.nan)}  base_A={_fmt(row["base_rbo_A"])}  base_B={_fmt(row["base_rbo_B"])}  global={_fmt(np.nan)}')
                print(f'          stoch: deterministic')
                results.append(row)
            else:
                print(f'  WARNING: coef_A.npy/coef_B.npy not found for LR pair {pid:02d}')


        # ─────────────────────────────────────────────────────────────
        # Explainer-based rows — one per explainer in EXPLAINERS
        # (skipped entirely for LR — no SHAP/LIME)
        # ─────────────────────────────────────────────────────────────
        for explainer_name in (EXPLAINERS if mt != 'logreg' else []):

            if explainer_name == 'shap':
                attr_dir = SHAP_DIR_BASE / mt / f'pair_{pid:02d}'
            elif explainer_name == 'lime':
                attr_dir = LIME_DIR_BASE / mt / f'pair_{pid:02d}'
            else:
                print(f'  WARNING: unknown explainer {explainer_name!r}')
                continue

            loaded = _load_attributions(explainer_name, pair_dir, attr_dir, pred_data)
            if loaded is None:
                print(f'  WARNING: {explainer_name} files not found for pair {pid:02d}. Skipping.')
                continue
            attr_A, attr_B, subset = loaded

            _validate_attribution_shapes(
                attr_A, attr_B, subset, flagged_local_idx,
                mt, explainer_name, pid
            )

            row = dict(base_row)
            row['explainer'] = explainer_name
            row['n_subset']  = int(len(subset))

            stoch_path = attr_dir / 'stochasticity.json'
            if stoch_path.exists():
                with open(stoch_path) as f:
                    stoch_data = json.load(f)

                row['explainer_stoch_max_abs']    = float(stoch_data.get('max_abs_diff', np.nan))
                row['explainer_stoch_median_abs'] = float(stoch_data.get('median_abs_diff', np.nan))
                row['explainer_is_deterministic'] = bool(stoch_data.get('is_deterministic', False))

                # Backwards-compatible alias for older plotting code / old result files.
                row['shap_stochasticity'] = row['explainer_stoch_max_abs']
            else:
                row['explainer_stoch_max_abs']    = np.nan
                row['explainer_stoch_median_abs'] = np.nan
                row['explainer_is_deterministic'] = np.nan
                row['shap_stochasticity']         = np.nan
                print(f'  WARNING: stochasticity.json not found under {attr_dir.name}')

            if attr_A.shape[1] == 0:
                for k in ['loc_cos', 'loc_rbo',
                          'base_cos_A', 'base_cos_B', 'base_rbo_A', 'base_rbo_B',
                          'global_rbo']:
                    row[k] = np.nan
                print(f'  [{explainer_name}] No subset instances — drift metrics NaN.')
                results.append(row)
                continue

            metrics = _attribution_drift_metrics(attr_A, attr_B)
            # Global drift is reported for SHAP only. LIME's local-surrogate
            # construction makes a mean-|attribution| ranking less directly
            # meaningful, so it is intentionally not reported.
            if explainer_name == 'lime':
                metrics['global_rbo'] = np.nan
            row.update(metrics)

            print(f'  [{explainer_name:<4s}]  cos: loc={_fmt(row["loc_cos"])}  base_A={_fmt(row["base_cos_A"])}  base_B={_fmt(row["base_cos_B"])}')
            print(f'          rbo: loc={_fmt(row["loc_rbo"])}  base_A={_fmt(row["base_rbo_A"])}  base_B={_fmt(row["base_rbo_B"])}  global={_fmt(row["global_rbo"])}')
            print(f'          stoch: max={_fmt_sci(row["explainer_stoch_max_abs"])}  median={_fmt_sci(row["explainer_stoch_median_abs"])}')

            results.append(row)

    all_results[mt] = results

print('\n✓ All drift metrics computed.')

## Save results

In [ ]:
# ── Save per-model-type CSVs (all explainers combined per model type) ─────────
for mt in MODEL_TYPES:
    if mt not in all_results or not all_results[mt]:
        continue
    mt_results_dir = RESULTS_DIR_BASE / mt
    mt_results_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(all_results[mt]).to_csv(mt_results_dir / 'drift_metrics.csv', index=False)
    print(f'Saved {mt}/drift_metrics.csv  ({len(all_results[mt])} rows)')

# ── Combined CSV ──────────────────────────────────────────────────────────────
combined_df = pd.concat(
    [pd.DataFrame(all_results[mt]) for mt in MODEL_TYPES
     if mt in all_results and all_results[mt]],
    ignore_index=True,
)
combined_df.to_csv(COMBINED_DIR / 'drift_metrics_combined.csv', index=False)
print(f'Saved combined/drift_metrics_combined.csv  ({len(combined_df)} rows)')

# ── Display summary per (model_type, explainer) ───────────────────────────────
if not combined_df.empty:
    print('\n── Rows per (model_type, explainer) ──')
    print(combined_df.groupby(['model_type', 'explainer']).size().to_string())

    print('\n── Per-pair summary (xgboost, shap) ──')
    sub = combined_df.query("model_type == 'xgboost' and explainer == 'shap'")
    if not sub.empty:
        cols = [c for c in [
            'pair_id', 'step_label_A', 'step_label_B',
            'pr_auc_A', 'pr_auc_B', 'delta_perf', 'delta_X', 'delta_Y',
            'loc_cos', 'base_cos_A', 'base_cos_B',
            'loc_rbo', 'base_rbo_A', 'base_rbo_B',
            'global_rbo', 'explainer_stoch_max_abs', 'explainer_stoch_median_abs',
        ] if c in sub.columns]
        print(sub[cols].to_string(index=False))

# ── Backwards-compat alias ────────────────────────────────────────────────────
drift_df = combined_df.query("model_type == 'xgboost' and explainer == 'shap'").copy() \
           if not combined_df.empty else pd.DataFrame()

## Temporal diagnostic dashboard

Three figures, rendered side by side per the methodology:
1. **Context strip** — Δ_X, Δ_Y, Δ_perf in three panels.
2. **Drift dashboard** — for each post-hoc stream (XGBoost·SHAP, XGBoost·LIME,
   MLP-PLR·SHAP, MLP-PLR·LIME), local cross-window drift on top of the
   within-window noise envelope between baseline_A and baseline_B. Cosine on
   row 1, RBO on row 2.
3. **LR reference** — full-window coefficient trajectories for windows A and B
   side by side (top), and replica coefficient noise (bottom). The top panels
   are skipped if `coef_A_full.npy` / `coef_B_full.npy` are not on disk.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Contextual signals — Δ_X, Δ_Y, Δ_perf
#
# Three-panel context strip. Δ_X and Δ_Y are model-agnostic (data-only) and
# rendered with point-value annotations and ymin=0 so that microscopic
# variation does not look visually large. Δ_perf is per-model and rendered
# on a symmetric axis around 0 so that gain/loss is readable at a glance.
# ─────────────────────────────────────────────────────────────────────────

def plot_context_strip(combined_df, model_types, save_path=None):
    if combined_df is None or combined_df.empty:
        print('combined_df is empty — skipping context strip.')
        return

    ctx = (combined_df
           .drop_duplicates('pair_id')
           .sort_values('pair_id')
           .reset_index(drop=True))
    pair_ids = ctx['pair_id'].tolist()
    x = list(range(len(pair_ids)))
    x_labels = ctx['step_label_A'].tolist()

    perf_df = (combined_df
               .drop_duplicates(['model_type', 'pair_id'])
               .sort_values('pair_id'))

    COLORS  = {'xgboost': 'crimson',  'logreg': 'steelblue', 'mlp_plr': 'darkviolet'}
    MARKERS = {'xgboost': 'o',        'logreg': 's',         'mlp_plr': '^'}
    LINES   = {'xgboost': '-',        'logreg': '--',        'mlp_plr': '-.'}

    fig, (ax_dx, ax_dy, ax_perf) = plt.subplots(1, 3, figsize=(15, 3.8))

    # Δ_X
    ax_dx.plot(x, ctx['delta_X'].values, marker='o', linestyle='-',
               color='steelblue', linewidth=1.6, markersize=6)
    for xi, vi in zip(x, ctx['delta_X'].values):
        ax_dx.annotate(f'{vi:.4f}', (xi, vi), textcoords='offset points',
                       xytext=(0, 9), ha='center', fontsize=7,
                       color='steelblue')
    ax_dx.set_title('Covariate drift  Δ$_X$', fontsize=11)
    ax_dx.set_ylabel('Mean Wasserstein distance')
    ax_dx.set_ylim(bottom=0)

    # Δ_Y
    ax_dy.plot(x, ctx['delta_Y'].values, marker='s', linestyle='-',
               color='darkorange', linewidth=1.6, markersize=6)
    for xi, vi in zip(x, ctx['delta_Y'].values):
        ax_dy.annotate(f'{vi:.4f}', (xi, vi), textcoords='offset points',
                       xytext=(0, 9), ha='center', fontsize=7,
                       color='darkorange')
    ax_dy.set_title('Target drift  Δ$_Y$', fontsize=11)
    ax_dy.set_ylabel('|p$_+^A$ − p$_+^B$|')
    ax_dy.set_ylim(bottom=0)

    # Δ_perf — symmetric around 0
    perf_max = perf_df['delta_perf'].abs().max()
    if pd.isna(perf_max) or perf_max == 0:
        perf_max = 0.05
    y_lim = perf_max * 1.25

    for mt in model_types:
        dft = perf_df[perf_df['model_type'] == mt]
        if dft.empty:
            continue
        xl = [pair_ids.index(pid) for pid in dft['pair_id'].tolist()]
        ax_perf.plot(xl, dft['delta_perf'].values,
                     marker=MARKERS[mt], linestyle=LINES[mt],
                     color=COLORS[mt], linewidth=1.8, markersize=6,
                     label=mt)
    ax_perf.axhline(0.0, color='grey', linewidth=0.8, linestyle=':')
    ax_perf.set_ylim(-y_lim, y_lim)
    ax_perf.set_title('Performance change  Δ$_{perf}$', fontsize=11)
    ax_perf.set_ylabel('Δ PR-AUC (B − A)')
    ax_perf.legend(fontsize=8, loc='best')

    for ax in (ax_dx, ax_dy, ax_perf):
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=8)
        ax.grid(True, axis='y', alpha=0.3)

    plt.suptitle('Contextual drift signals across the timeline', fontsize=13, y=1.02)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved {Path(save_path).name}')
    plt.show()


plot_context_strip(combined_df, MODEL_TYPES,
                   save_path=COMBINED_DIR / 'drift_context_strip.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Drift dashboard — local cross-window drift vs within-window noise envelope
#
# 2 rows × 4 columns, one column per (model_type, explainer) stream:
#   XGBoost·SHAP, XGBoost·LIME, MLP-PLR·SHAP, MLP-PLR·LIME
# Row 1 = cosine, row 2 = RBO. The grey envelope spans
# [min(base_A, base_B), max(base_A, base_B)] and represents the within-window
# noise floor; the coloured line on top is the cross-window drift.
#
# A column is rendered as "(no data)" if its (model_type, explainer) stream is
# empty in combined_df (e.g. no LIME computed for a model).
# ─────────────────────────────────────────────────────────────────────────

def plot_drift_dashboard(combined_df, save_path=None):
    if combined_df is None or combined_df.empty:
        print('combined_df is empty — skipping drift dashboard.')
        return

    df = combined_df[combined_df['explainer'].isin(['shap', 'lime'])].copy()
    if df.empty:
        print('No SHAP/LIME rows in combined_df — skipping drift dashboard.')
        return

    streams = [
        ('xgboost', 'shap', 'XGBoost · SHAP',  '#b30000'),
        ('xgboost', 'lime', 'XGBoost · LIME',  '#fc8d59'),
        ('mlp_plr', 'shap', 'MLP-PLR · SHAP',  '#54278f'),
        ('mlp_plr', 'lime', 'MLP-PLR · LIME',  '#9e9ac8'),
    ]

    pair_ids = sorted(combined_df['pair_id'].unique())
    x = list(range(len(pair_ids)))
    label_lookup = (combined_df.drop_duplicates('pair_id')
                              .set_index('pair_id')['step_label_A'].to_dict())
    x_labels = [label_lookup[pid] for pid in pair_ids]

    cos_max = float(np.nanmax(df[['loc_cos', 'base_cos_A', 'base_cos_B']].values))
    rbo_max = float(np.nanmax(df[['loc_rbo', 'base_rbo_A', 'base_rbo_B']].values))
    cos_ymax = max(1.0, 1.05 * cos_max)
    rbo_ymax = max(1.0, 1.05 * rbo_max)

    fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 7), sharey='row')

    for col, (mt, expl, title, color) in enumerate(streams):
        sub = (df[(df['model_type'] == mt) & (df['explainer'] == expl)]
               .sort_values('pair_id'))

        if sub.empty:
            for row in (0, 1):
                axes[row, col].set_title(f'{title}\n(no data)', fontsize=10)
                axes[row, col].set_xticks([])
                axes[row, col].grid(False)
            continue

        xl = [pair_ids.index(pid) for pid in sub['pair_id'].tolist()]

        # Cosine row
        ax = axes[0, col]
        bA = sub['base_cos_A'].values
        bB = sub['base_cos_B'].values
        ax.fill_between(xl, np.minimum(bA, bB), np.maximum(bA, bB),
                        color='lightgray', alpha=0.7, zorder=1,
                        edgecolor='gray', linewidth=0.3)
        ax.plot(xl, sub['loc_cos'].values, marker='o', linestyle='-',
                color=color, linewidth=2.4, markersize=6, zorder=3)
        ax.set_title(title, fontsize=10.5, color=color, fontweight='bold')
        ax.set_ylim(0, cos_ymax)
        ax.set_xticks(x)
        ax.set_xticklabels([] * len(x))
        ax.grid(True, axis='y', alpha=0.3)
        if col == 0:
            ax.set_ylabel('Cosine distance', fontsize=10.5)

        # RBO row
        ax = axes[1, col]
        bA = sub['base_rbo_A'].values
        bB = sub['base_rbo_B'].values
        ax.fill_between(xl, np.minimum(bA, bB), np.maximum(bA, bB),
                        color='lightgray', alpha=0.7, zorder=1,
                        edgecolor='gray', linewidth=0.3)
        ax.plot(xl, sub['loc_rbo'].values, marker='o', linestyle='-',
                color=color, linewidth=2.4, markersize=6, zorder=3)
        ax.set_ylim(0, rbo_ymax)
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7.5)
        ax.set_xlabel('Window pair (A end date)', fontsize=9)
        ax.grid(True, axis='y', alpha=0.3)
        if col == 0:
            ax.set_ylabel('RBO distance (1 − RBO)', fontsize=10.5)

    legend_handles = [
        Line2D([0], [0], color='black', linewidth=2.4, marker='o',
               markersize=6, label='Cross-window drift  (loc)'),
        Patch(facecolor='lightgray', edgecolor='gray', alpha=0.9,
              label='Within-window noise envelope  [base$_A$, base$_B$]'),
    ]
    fig.legend(handles=legend_handles, loc='upper center',
               bbox_to_anchor=(0.5, 1.015), ncol=2, fontsize=10,
               frameon=False)

    plt.suptitle('Post-hoc explanation drift — cross-window distance vs within-window noise',
                 fontsize=13, y=1.06)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved {Path(save_path).name}')
    plt.show()


plot_drift_dashboard(combined_df,
                     save_path=COMBINED_DIR / 'drift_dashboard.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Logistic regression — transparent coefficient reference
#
# Two-section figure dedicated to the simple, interpretable model:
#
# Section 1: Coefficient trajectories. Two side-by-side panels (Window A,
# Window B). The top-K features by max-across-pairs |coef_A − coef_B| are
# plotted as one line per feature; same colour-per-feature on both sides
# so the eye can track the same feature across panels. Shared y-axis.
# Requires coef_A_full.npy / coef_B_full.npy on disk; if missing, this
# section is skipped and a note is printed.
#
# Section 2: LR replica noise. Single panel with cosine and RBO baselines
# on windows A and B. Quantifies how reproducible LR's coefficients are
# across replicas — directly parallel to the post-hoc dashboard's noise
# band, but standing on its own (LR has no cross-window drift score).
#
# Caveat: each pair's A and B fit uses its own scaler, so coefficient
# absolute magnitudes between A and B are not strictly commensurable; sign
# and qualitative shape are. This applies to section 1 only.
# ─────────────────────────────────────────────────────────────────────────

def plot_lr_reference(combined_df, pairs, feature_names, model_dir_base,
                      top_k=10, save_path=None):
    lr_dir_base = Path(model_dir_base) / 'logreg'

    available_pairs = []
    coef_A_list, coef_B_list = [], []
    for p in pairs:
        pid = p['pair_id']
        a_path = lr_dir_base / f'pair_{pid:02d}' / 'coef_A_full.npy'
        b_path = lr_dir_base / f'pair_{pid:02d}' / 'coef_B_full.npy'
        if not (a_path.exists() and b_path.exists()):
            continue
        coef_A_list.append(np.load(a_path))
        coef_B_list.append(np.load(b_path))
        available_pairs.append(p)

    have_coefs = len(available_pairs) > 0
    if not have_coefs:
        print('No full-window LR coefficients found on disk — section 1 will be skipped.')
        print('  (To enable: have notebook 02 also write coef_A_full.npy and coef_B_full.npy '
              'as deterministic full-window LR fits.)')

    lr_rows = (combined_df[combined_df['model_type'] == 'logreg']
               .sort_values('pair_id')
               .reset_index(drop=True))
    have_noise = not lr_rows.empty

    if not (have_coefs or have_noise):
        print('No LR data of any kind — skipping figure.')
        return

    if have_coefs and have_noise:
        fig = plt.figure(figsize=(14, 9))
        gs = fig.add_gridspec(2, 2, height_ratios=[1.2, 1.0], hspace=0.5,
                              wspace=0.18)
        ax_traj_A = fig.add_subplot(gs[0, 0])
        ax_traj_B = fig.add_subplot(gs[0, 1], sharey=ax_traj_A)
        ax_noise  = fig.add_subplot(gs[1, :])
    elif have_coefs:
        fig, (ax_traj_A, ax_traj_B) = plt.subplots(1, 2, figsize=(14, 4.5),
                                                    sharey=True)
        ax_noise = None
    else:
        fig, ax_noise = plt.subplots(1, 1, figsize=(11, 4.0))
        ax_traj_A = ax_traj_B = None

    # Section 1: trajectories
    if have_coefs:
        coef_A_stack = np.array(coef_A_list)
        coef_B_stack = np.array(coef_B_list)
        gap = np.abs(coef_A_stack - coef_B_stack)
        feat_score = gap.max(axis=0)
        top_idx = np.argsort(-feat_score)[:top_k]

        n_pairs_lr = len(available_pairs)
        x_traj = np.arange(n_pairs_lr)
        x_traj_labels = [p['step_label_A'] for p in available_pairs]

        cmap = plt.get_cmap('tab20')
        colors_per_feat = [cmap(i / max(top_k - 1, 1) * 0.95) for i in range(top_k)]

        for ax, stack, side_label in [(ax_traj_A, coef_A_stack, 'Window A'),
                                       (ax_traj_B, coef_B_stack, 'Window B')]:
            for j_local, feat_idx in enumerate(top_idx):
                ax.plot(x_traj, stack[:, feat_idx],
                        marker='o', markersize=4, linewidth=1.5,
                        color=colors_per_feat[j_local],
                        label=feature_names[feat_idx] if ax is ax_traj_B else None)
            ax.axhline(0.0, color='grey', linewidth=0.6, linestyle=':')
            ax.set_title(f'{side_label} — full-window LR coefficients', fontsize=11)
            ax.set_xticks(x_traj)
            ax.set_xticklabels(x_traj_labels, rotation=30, ha='right', fontsize=7.5)
            ax.set_xlabel('Window pair (A end date)', fontsize=9)
            ax.grid(True, axis='y', alpha=0.3)

        ax_traj_A.set_ylabel('Standardised coefficient', fontsize=10.5)
        ax_traj_B.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
                         fontsize=8, frameon=False,
                         title=f'Top {top_k} features by |Δcoef|',
                         title_fontsize=8.5)

    # Section 2: replica noise
    if have_noise and ax_noise is not None:
        x_noise = np.arange(len(lr_rows))
        x_noise_labels = lr_rows['step_label_A'].tolist()

        ax_noise.plot(x_noise, lr_rows['base_cos_A'].values, marker='o', linestyle='-',
                      color='steelblue', linewidth=1.6, label='cos · base$_A$')
        ax_noise.plot(x_noise, lr_rows['base_cos_B'].values, marker='s', linestyle='--',
                      color='steelblue', linewidth=1.6, label='cos · base$_B$', alpha=0.7)
        ax_noise.plot(x_noise, lr_rows['base_rbo_A'].values, marker='o', linestyle='-',
                      color='darkorange', linewidth=1.6, label='RBO · base$_A$')
        ax_noise.plot(x_noise, lr_rows['base_rbo_B'].values, marker='s', linestyle='--',
                      color='darkorange', linewidth=1.6, label='RBO · base$_B$', alpha=0.7)

        ax_noise.set_title('LR replica coefficient noise (within-window instability)',
                           fontsize=11)
        ax_noise.set_xticks(x_noise)
        ax_noise.set_xticklabels(x_noise_labels, rotation=30, ha='right', fontsize=8)
        ax_noise.set_xlabel('Window pair (A end date)', fontsize=9)
        ax_noise.set_ylabel('Distance', fontsize=10.5)
        ax_noise.set_ylim(bottom=0)
        ax_noise.grid(True, axis='y', alpha=0.3)
        ax_noise.legend(fontsize=8, loc='best', ncol=2)

    plt.suptitle('Logistic regression — transparent coefficient reference',
                 fontsize=13, y=1.0)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved {Path(save_path).name}')
    plt.show()


plot_lr_reference(combined_df, pairs, feature_names, MODEL_DIR_BASE,
                  top_k=10,
                  save_path=COMBINED_DIR / 'lr_reference.png')

## Per-feature SHAP drift profile

Shows which features shift most in global importance across the timeline.

In [ ]:
# Per-feature global SHAP attribution over time. Default: XGBoost.
REPORT_MT = 'xgboost'

attr_dir_mt = SHAP_DIR_BASE / REPORT_MT

global_imp_matrix = []
pair_labels = []
for p in pairs:
    pid = p['pair_id']
    ap  = attr_dir_mt / f'pair_{pid:02d}' / 'shap_A.npy'
    if not ap.exists():
        continue
    attr_A = np.load(ap)        # (R, |F|, p)
    phi_bar_A = attr_A.mean(axis=0)
    g_A = np.abs(phi_bar_A).mean(axis=0)
    global_imp_matrix.append(g_A)
    pair_labels.append(p['step_label_A'])

if global_imp_matrix:
    imp_arr = np.array(global_imp_matrix)
    feat_variance = imp_arr.std(axis=0)
    top_feat_idx = np.argsort(-feat_variance)[:15]

    fig, ax = plt.subplots(figsize=(12, 5))
    for j in top_feat_idx:
        ax.plot(range(len(pair_labels)), imp_arr[:, j], marker='o',
                label=feature_names[j], linewidth=1.5)
    ax.set_title(f'Global SHAP importance over time — top 15 variable features  ({REPORT_MT})')
    ax.set_xlabel('Window pair (A end date)')
    ax.set_ylabel('Mean |SHAP|')
    ax.set_xticks(range(len(pair_labels)))
    ax.set_xticklabels(pair_labels, rotation=30, ha='right', fontsize=7)
    ax.legend(loc='upper right', fontsize=7, ncol=2)
    plt.tight_layout()
    out_dir = RESULTS_DIR_BASE / REPORT_MT
    out_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_dir / 'feature_importance_over_time_shap.png', dpi=150)
    plt.show()
else:
    print(f'No SHAP attribution files found for {REPORT_MT}.')

## Summary statistics

In [ ]:
numeric_cols = [
    'pr_auc_A', 'pr_auc_B', 'delta_perf',
    'delta_X', 'delta_Y',
    'loc_cos', 'base_cos_A', 'base_cos_B',
    'loc_rbo', 'base_rbo_A', 'base_rbo_B',
    'global_rbo', 'explainer_stoch_max_abs', 'explainer_stoch_median_abs',
]

if not combined_df.empty:
    for (mt, exp), group_df in combined_df.groupby(['model_type', 'explainer']):
        avail_cols = [c for c in numeric_cols if c in group_df.columns]
        summary = group_df[avail_cols].describe().T[['mean', 'std', 'min', 'max']]
        print(f'\n=== {mt} / {exp} — overall summary ({len(group_df)} pairs) ===')
        print(summary.round(4).to_string())

        # LR rows show NaN for cross-window drift columns by design — they are a
        # transparent reference, not a post-hoc explanation drift stream.

        # Side-by-side mean local drift vs mean baselines (no ratio).
        # Helps eyeball whether dynamic drift sits above the within-window noise floor.
        if all(c in group_df.columns for c in
               ['loc_cos', 'base_cos_A', 'base_cos_B', 'loc_rbo', 'base_rbo_A', 'base_rbo_B']):
            print(
                f"  mean loc_cos = {group_df['loc_cos'].mean():.4f}  |  "
                f"mean baseline_A = {group_df['base_cos_A'].mean():.4f}  |  "
                f"mean baseline_B = {group_df['base_cos_B'].mean():.4f}"
            )
            print(
                f"  mean loc_rbo = {group_df['loc_rbo'].mean():.4f}  |  "
                f"mean baseline_A = {group_df['base_rbo_A'].mean():.4f}  |  "
                f"mean baseline_B = {group_df['base_rbo_B'].mean():.4f}"
            )